# 🎫 Support Ticket Classifier
### NLP Pipeline · Category Detection · Priority Tagging · Model Evaluation
---
**Stack**: Python · Scikit-learn · TF-IDF · Logistic Regression  
**Author**: Portfolio Project — Full Stack Development

In [ ]:
import sys
sys.path.insert(0, 'src')

import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import Counter

from data_generator import generate_dataset
from preprocessor import TicketPreprocessor
from classifier import TicketClassifier

print('✅ All modules loaded')

## 1. Generate Synthetic Dataset

In [ ]:
tickets = generate_dataset(1200, seed=42)
print(f'Generated {len(tickets)} tickets\n')

# Distribution stats
cat_dist = Counter(t['category'] for t in tickets)
pri_dist = Counter(t['priority'] for t in tickets)

print('Category distribution:')
for k, v in sorted(cat_dist.items()):
    bar = '█' * (v // 10)
    print(f'  {k:20s} {v:4d}  {bar}')

print('\nPriority distribution:')
for k, v in [('high', pri_dist['high']), ('medium', pri_dist['medium']), ('low', pri_dist['low'])]:
    bar = '█' * (v // 10)
    print(f'  {k:10s} {v:4d}  {bar}')

## 2. Text Preprocessing Pipeline

In [ ]:
preprocessor = TicketPreprocessor()

sample_tickets = [
    "I was charged $299 twice this month! Please refund IMMEDIATELY — this is urgent!",
    "Hi there, can't access my account for 3 days. It's blocking our team's work.",
    "Just wondering when my subscription renews. Thanks!",
]

print('=== Text Preprocessing Pipeline ===\n')
for ticket in sample_tickets:
    result = preprocessor.process(ticket)
    print(f'ORIGINAL : {result["original"]}')
    print(f'CLEANED  : {result["cleaned"]}')
    print(f'TOKENS   : {result["tokens"][:10]}')
    print(f'FEATURES : urgency_high={result["features"]["urgency_high_score"]}  '
          f'exclamations={result["features"]["exclamation_count"]}')
    print()

## 3. Train the Classifier

In [ ]:
clf = TicketClassifier()
metrics = clf.fit(tickets)
clf.save('models/classifier.pkl')

print('\n=== Model Performance Summary ===')
for task in ['category', 'priority']:
    m = metrics[task]
    print(f'\n{task.upper()}:')
    print(f'  Accuracy    : {m["accuracy"]:.1%}')
    print(f'  F1 (wtd)    : {m["f1_weighted"]:.1%}')
    print(f'  CV F1 mean  : {m["cv_f1_mean"]:.1%} ± {m["cv_f1_std"]:.3f}')

## 4. Visualise Results

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Support Ticket Classifier — Analytics', fontsize=14, fontweight='bold')

# ── 1. Category distribution pie ─────────────────────────────────────────────
ax = axes[0]
colors = ['#3B82F6', '#10B981', '#F59E0B', '#EF4444', '#8B5CF6']
ax.pie(cat_dist.values(), labels=cat_dist.keys(), colors=colors,
       autopct='%1.0f%%', startangle=90)
ax.set_title('Category Distribution')

# ── 2. Priority bar chart ─────────────────────────────────────────────────────
ax = axes[1]
pri_colors = {'high': '#EF4444', 'medium': '#F59E0B', 'low': '#10B981'}
pris = ['high', 'medium', 'low']
vals = [pri_dist[p] for p in pris]
bars = ax.bar(pris, vals, color=[pri_colors[p] for p in pris], edgecolor='white', linewidth=2)
for bar, val in zip(bars, vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
            str(val), ha='center', fontweight='bold')
ax.set_title('Priority Distribution')
ax.set_ylabel('Ticket Count')

# ── 3. Model accuracy comparison ─────────────────────────────────────────────
ax = axes[2]
tasks = ['Category\nClassifier', 'Priority\nClassifier']
accuracies = [metrics['category']['accuracy'], metrics['priority']['accuracy']]
f1s = [metrics['category']['f1_weighted'], metrics['priority']['f1_weighted']]
x = np.arange(len(tasks))
width = 0.35
ax.bar(x - width/2, accuracies, width, label='Accuracy', color='#3B82F6')
ax.bar(x + width/2, f1s, width, label='F1 (weighted)', color='#10B981')
ax.set_ylim(0, 1.1)
ax.set_xticks(x)
ax.set_xticklabels(tasks)
ax.set_title('Model Performance')
ax.set_ylabel('Score')
ax.legend()
for i, (acc, f1) in enumerate(zip(accuracies, f1s)):
    ax.text(i - width/2, acc + 0.01, f'{acc:.1%}', ha='center', fontsize=9)
    ax.text(i + width/2, f1 + 0.01, f'{f1:.1%}', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('outputs/analytics.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved → outputs/analytics.png')

## 5. Live Predictions

In [ ]:
PRIORITY_ICONS = {'high': '🔴', 'medium': '🟡', 'low': '🟢'}
CAT_ICONS = {
    'billing': '💳', 'technical': '⚙️', 'account': '👤',
    'general_inquiry': '❓', 'shipping': '📦'
}

test_tickets = [
    "Our entire production database is DOWN. All 500 users affected. Losing $10k/hour!",
    "I was charged twice this month — $299 x2. Please refund ASAP.",
    "The PDF export produces blank pages for reports over 10 pages.",
    "Account hacked — password was changed without my knowledge. LOCK IT!",
    "Can I get a receipt for last month's payment? For my tax records.",
    "Do you offer a student discount?",
    "My order tracking hasn't updated in 5 days — event is tomorrow!",
]

print(f'{'─'*70}')
print(f'{'TICKET TEXT':45s}  {'CATEGORY':18s}  PRIORITY')
print(f'{'─'*70}')

for text in test_tickets:
    result = clf.predict(text)
    cat = result['category']
    pri = result['priority']
    cat_icon = CAT_ICONS.get(cat, '❓')
    pri_icon = PRIORITY_ICONS.get(pri, '⚪')
    short = text[:43] + '..' if len(text) > 45 else text
    print(f'{short:45s}  {cat_icon} {cat:15s}  {pri_icon} {pri}')

## 6. Top Predictive Features

In [ ]:
print('=== Top Features: Category Model ===')
cat_features = clf.top_features('category', 8)
for cat, feats in cat_features.items():
    words = [f[0] for f in feats]
    print(f'  {cat:20s}: {words}')

print('\n=== Top Features: Priority Model ===')
pri_features = clf.top_features('priority', 8)
for pri, feats in pri_features.items():
    words = [f[0] for f in feats]
    print(f'  {pri:10s}: {words}')